# 00. 威胁建模与 OWASP Top 10

OWASP Top 10 for LLM Applications 2025 是 Agent 安全的基础框架。其中 **5 个风险对 Agent 尤其关键**：

| ID | 风险 | Agent 关键性 |
|----|------|-------------|
| LLM01 | Prompt Injection | ⭐⭐⭐ 直接+间接注入 |
| LLM05 | Improper Output Handling | ⭐⭐⭐ 工具输出未净化 |
| LLM06 | Excessive Agency | ⭐⭐⭐ 权限过大 |
| LLM07 | System Prompt Leakage | ⭐⭐ 指令泄露 |
| LLM08 | Vector/Embedding Weakness | ⭐⭐ RAG 投毒 |

> **检查点**：能说出 Agent 为什么比普通 LLM App 更易受 LLM05 和 LLM06 攻击。

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'xmake.lua').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'agents'))

from scripts.nb_helpers import setup, chat, chat_json, chat_with_tools, show_trace
client = setup()


🌐 API 模式 — model=deepseek-v4-flash  base_url=https://api.deepseek.com


In [2]:
# 威胁建模：给一个 Agent 做风险评估
from dataclasses import dataclass, field

@dataclass
class Threat:
    owasp_id: str; risk: str; likelihood: str; impact: str
    vector: str; agent_specific: bool

def model_threats(agent_config: dict) -> list[Threat]:
    threats = []
    if agent_config.get("has_user_input"):
        threats.append(Threat("LLM01","Prompt Injection","High","High",
            "User input hijacks agent behavior", True))
    if agent_config.get("has_tools"):
        threats.append(Threat("LLM05","Improper Output Handling","High","Critical",
            "Tool output executed without sanitization", True))
        threats.append(Threat("LLM06","Excessive Agency","Medium","Critical",
            f"Agent has {agent_config.get('tool_count',0)} tools with broad permissions", True))
    if agent_config.get("has_rag"):
        threats.append(Threat("LLM08","Vector Weakness","Medium","Medium",
            "Poisoned documents in RAG corpus", True))
    threats.append(Threat("LLM02","Sensitive Disclosure","Medium","High",
        "Agent context leaks PII", False))
    return threats

# 评估一个「文件管理 Agent」
config = {"has_user_input":True,"has_tools":True,"tool_count":5,"has_rag":False,"has_file_access":True}
threats = model_threats(config)

print("Threat Model: File Manager Agent\n")
for t in threats:
    agent_flag = "[AGENT]" if t.agent_specific else "[GEN]"
    severity = "CRITICAL" if t.impact=="Critical" and t.likelihood=="High" else f"{t.likelihood}/{t.impact}"
    print(f"  {agent_flag} {t.owasp_id} ({severity}): {t.risk}")
    print(f"    Vector: {t.vector}")

Threat Model: File Manager Agent

  [AGENT] LLM01 (High/High): Prompt Injection
    Vector: User input hijacks agent behavior
  [AGENT] LLM05 (CRITICAL): Improper Output Handling
    Vector: Tool output executed without sanitization
  [AGENT] LLM06 (Medium/Critical): Excessive Agency
    Vector: Agent has 5 tools with broad permissions
  [GEN] LLM02 (Medium/High): Sensitive Disclosure
    Vector: Agent context leaks PII


## 风险矩阵

```
Impact
  ↑ Critical  │ LLM05 LLM06
  │ High      │ LLM01 LLM02
  │ Medium    │ LLM08 LLM07
  │ Low       │ LLM04 LLM09 LLM10
  └────────────────────→ Likelihood
      Low   Med   High
```

> **检查点**：能为你的 Agent 画出风险矩阵，标注至少 3 个威胁。

## 真实 API 实验

让真实 LLM 对一个 agent 配置做威胁建模（`threat_model_demo.py --use-api`），对比静态映射表。


In [3]:
SCRIPT = ROOT / 'agents' / 'safety' / 'scripts' / 'threat_model_demo.py'
import subprocess, sys
result = subprocess.run([sys.executable, str(SCRIPT), '--agent-config', 'privileged',
                         '--attack', 'excessive-agency', '--use-api'], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr, file=sys.stderr)


Config: privileged = {'tools': ['search', 'shell', 'file_write'], 'memory': True, 'internet': True, 'human_approval': False}
Attack: excessive-agency

[LLM08] agent 权限过大，自主执行不可逆操作
攻击面分析:
  - 高危工具=['shell', 'file_write']
  - human_approval=False
风险等级: HIGH

=== LLM threat analysis ===
**攻击路径（3步以内）：**  
1. 攻击者通过提示注入使Agent执行`shell`命令，如`rm -rf /`。  
2. Agent调用`search`获取内部敏感文档，结合`file_write`将数据外泄到公网文件。  
3. Agent使用`memory`持久化恶意指令，后续无需再次注入即可反复执行危险操作。  

**影响：**  
- 系统完全接管（任意命令执行）  
- 数据泄露（读取/写入任意文件，外传机密）  
- 持久化后门（memory存储恶意操作，持续控制）  

**2条缓解措施：**  
1. **最小权限工具约束**：对`shell`工具仅允许白名单命令（如`ls`、`cat`），禁止删除、写入、网络操作；对`file_write`限制可写路径（如仅`/tmp/`）和文件类型。  
2. **强制人类审批**：对于所有高风险工具（`shell`、`file_write`），即使`human_approval=False`，需在代码层强制增加人工确认步骤（如二次确认框或审查队列），或对`memory`的写入操作增加权限校验。



## 练习

1. 为你正在开发（或计划开发）的 Agent 做一次完整的威胁建模。列出所有 OWASP 风险。
2. 设计一个「最小权限」的 Agent 工具权限配置：只给完成任务必需的权限。
3. 调研一个真实的 Agent 安全事故（如 indirect prompt injection），分析根因和防御措施。